# DNA Methylation Data - Exploration and Data Analysis

In [1]:
import pandas as pd

In [2]:
dna_meth = pd.read_csv('./data/BRCA.DNAmethy_filtered.csv', sep=';')
print("The BRCA.DNAmethy_filtered.csv file has ", dna_meth.shape[0], "CpGs islands values and ", dna_meth.shape[1], "TCGA patients")

The BRCA.DNAmethy_filtered.csv file has  486427 CpGs islands values and  786 TCGA patients


The BRCA.DNAmethy_filtered.csv file has **486 427** CpGs islands values and **786** TCGA patients. </br>

Patients were sequenced using Illumina Human methylation 450

In [3]:
dna_meth.columns

Index(['CpG_ID', 'TCGA.3C.AAAU', 'TCGA.3C.AALI', 'TCGA.3C.AALJ',
       'TCGA.3C.AALK', 'TCGA.4H.AAAK', 'TCGA.5L.AAT0', 'TCGA.5L.AAT1',
       'TCGA.5T.A9QA', 'TCGA.A1.A0SB',
       ...
       'TCGA.UL.AAZ6', 'TCGA.UU.A93S', 'TCGA.V7.A7HQ', 'TCGA.W8.A86G',
       'TCGA.WT.AB41', 'TCGA.WT.AB44', 'TCGA.XX.A899', 'TCGA.XX.A89A',
       'TCGA.Z7.A8R5', 'TCGA.Z7.A8R6'],
      dtype='object', length=786)

In [4]:
dna_meth = dna_meth.set_index('CpG_ID')

We need to filter the table for NaNs, so we will look at the CpG islands that have more than 99% of data completness

In [5]:
# Step 1: Calculate the percentage of non-missing values per CpG site (row)
completeness_per_cpg = dna_meth.notna().sum(axis=1) / dna_meth.shape[1]

# Step 2: Keep only CpGs with at least 99% of data present
threshold = 0.99
cpgs_to_keep = completeness_per_cpg >= threshold

print(f"Original number of CpG sites: {len(dna_meth)}")
print(f"CpG sites with ≥99% data: {cpgs_to_keep.sum()}")
print(f"CpG sites removed: {(~cpgs_to_keep).sum()}")

# Step 3: Filter the DataFrame
dna_meth_filtered = dna_meth[cpgs_to_keep]

# Check remaining NaN values
remaining_nans = dna_meth_filtered.isna().sum().sum()
print(f"\nRemaining NaN values: {remaining_nans}")

Original number of CpG sites: 486427
CpG sites with ≥99% data: 356801
CpG sites removed: 129626

Remaining NaN values: 129614


For the rest of the NaN value, we can impute them via 4 approaches : 
- We drop them (the strictest)
- We impute with the mean
- We impute with the median
- we fill with zero (but introduce biological bias...)

In [6]:
dna_meth_clean = dna_meth.dropna(axis=0)  # Remove rows with any NaN

# # Fill NaN with the mean value of that CpG site across all patients
# dna_meth_clean = dna_meth.T.fillna(dna_meth.mean(axis=1)).T

# # Fill NaN with the median value of that CpG site
# dna_meth_clean = dna_meth.T.fillna(dna_meth.median(axis=1)).T

# # If zero methylation is biologically meaningful
# dna_meth_clean = dna_meth.fillna(0)

### DNA Methylation Preprocessing

<ol>
    <li> Enlever les probes (CpG) pour lesquels le signal = 0 (Y-a-t-il une réduction de la variance ?) (OK)
    <li> ANOVA F-test (one way test - ANOVA à un facteur), MAIS ATTENTION UTILISER QUE LES DONNÉES ENTRAINEMENT
    <li> FDR Transformation (Benjamini-Hochberg procedure)
    <li> Enlever les valeurs manquantes, et ne garder que les probes avec au moins 99% de données présentes (Déjà ok)
</ol>

### Load Clinical Data and Prepare Labels for ANOVA

We'll use the survival groups from the clinical data to perform ANOVA F-test.

In [18]:
# Load clinical data with survival group labels
clinical_data = pd.read_csv('data/BRCA.clinical.csv')

print(f"Total patients in clinical data: {len(clinical_data)}")
print(f"Survival group distribution:")
print(clinical_data['OS.survival_group'].value_counts())

# Convert patient IDs from "-" format to "." format to match methylation data
clinical_data['patient_id_dots'] = clinical_data['bcr_patient_barcode'].str.replace('-', '.')

# Filter out "not considered" patients for ANOVA analysis
clinical_filtered = clinical_data[clinical_data['OS.survival_group'] != 'not considered'].copy()

print(f"\nPatients after filtering 'not considered': {len(clinical_filtered)}")
print(f"Survival group distribution (filtered):")
print(clinical_filtered['OS.survival_group'].value_counts())

# Find patients that are in both clinical data and methylation data
common_patients = list(set(clinical_filtered['patient_id_dots']) & set(dna_meth_clean.columns))

print(f"\nPatients present in both clinical and methylation data: {len(common_patients)}")

# Filter methylation data to only include these patients
dna_meth_for_anova = dna_meth_clean[common_patients]

# Create labels aligned with the methylation data columns
labels_df = clinical_filtered.set_index('patient_id_dots').loc[common_patients]
labels = labels_df['OS.survival_group']

print(f"\nFinal dataset for ANOVA:")
print(f"Methylation data shape: {dna_meth_for_anova.shape}")
print(f"Labels shape: {len(labels)}")
print(f"Label distribution: {labels.value_counts().to_dict()}")



Total patients in clinical data: 1097
Survival group distribution:
OS.survival_group
not considered    774
long              251
short              72
Name: count, dtype: int64

Patients after filtering 'not considered': 323
Survival group distribution (filtered):
OS.survival_group
long     251
short     72
Name: count, dtype: int64

Patients present in both clinical and methylation data: 230

Final dataset for ANOVA:
Methylation data shape: (301291, 230)
Labels shape: 230
Label distribution: {'long': 179, 'short': 51}


In [19]:
from sklearn.feature_selection import f_classif, SelectKBest
from statsmodels.stats.multitest import multipletests
import numpy as np

# Prepare data for ANOVA
# X: samples as rows, features (CpGs) as columns
X = dna_meth_for_anova.T  

# Ensure labels are in the same order as X
y = labels[X.index]

print(f"X shape: {X.shape} (patients × CpG sites)")
print(f"y shape: {y.shape}")
print(f"Labels: {y.value_counts().to_dict()}")

# Perform ANOVA F-test
f_scores, p_values = f_classif(X, y)

# Apply Benjamini-Hochberg FDR correction
reject, p_values_corrected, _, _ = multipletests(
    p_values, 
    alpha=0.05, 
    method='fdr_bh'
)

# Create a DataFrame with results
anova_results = pd.DataFrame({
    'CpG_ID': dna_meth_for_anova.index,
    'F_score': f_scores,
    'p_value': p_values,
    'p_value_corrected': p_values_corrected,
    'significant_fdr': reject
})

# Sort by p-value
anova_results = anova_results.sort_values(by='p_value', ascending=True)

print(f"\nNumber of CpG sites tested: {len(anova_results)}")
print(f"Significant CpGs (p < 0.05, uncorrected): {(anova_results['p_value'] < 0.05).sum()}")
print(f"Significant CpGs (FDR corrected): {anova_results['significant_fdr'].sum()}")
print(f"\nTop 10 most significant CpG sites:")
print(anova_results.head(10))



X shape: (230, 301291) (patients × CpG sites)
y shape: (230,)
Labels: {'long': 179, 'short': 51}

Number of CpG sites tested: 301291
Significant CpGs (p < 0.05, uncorrected): 35922
Significant CpGs (FDR corrected): 450

Top 10 most significant CpG sites:
            CpG_ID    F_score       p_value  p_value_corrected  \
168905  cg14479910  45.215188  1.403056e-10           0.000042   
226196  cg20175418  36.538297  6.055700e-09           0.000912   
184774  cg16040341  32.754115  3.272792e-08           0.002623   
217670  cg19350024  32.615901  3.482752e-08           0.002623   
144956  cg12379948  31.984014  4.630222e-08           0.002763   
80997   cg06563873  31.602131  5.502061e-08           0.002763   
99495   cg08116711  31.249387  6.454345e-08           0.002778   
132225  cg11142826  30.898546  7.566937e-08           0.002850   
67405   cg05357108  30.562803  8.813005e-08           0.002898   
216786  cg19253831  30.355754  9.682877e-08           0.002898   

        significan

In [20]:
# Filter CpG sites using FDR-corrected significance
# Option 1: Use FDR-corrected results (recommended)
significant_cpgs = anova_results.loc[anova_results['significant_fdr'], 'CpG_ID']

# Option 2: Use uncorrected p-value < 0.05 (more permissive)
# significant_cpgs = anova_results.loc[anova_results['p_value'] < 0.05, 'CpG_ID']

# Filter the full cleaned methylation data to only include significant CpGs
dna_meth_significant = dna_meth_clean.loc[significant_cpgs]

print(f"Original CpG sites: {dna_meth_clean.shape[0]}")
print(f"Significant CpG sites: {dna_meth_significant.shape[0]}")
print(f"Patients: {dna_meth_significant.shape[1]}")
print(f"\nReduction: {(1 - dna_meth_significant.shape[0]/dna_meth_clean.shape[0])*100:.2f}% of CpG sites removed")

Original CpG sites: 301291
Significant CpG sites: 450
Patients: 785

Reduction: 99.85% of CpG sites removed


In [21]:
dna_meth_significant.shape

(450, 785)

### Summary of ANOVA-based Feature Selection

The ANOVA F-test was performed using survival group labels (short vs long) from the clinical data:
1. Loaded clinical data and extracted OS.survival_group labels
2. Filtered out "not considered" patients
3. Matched patients between clinical and methylation data
4. Performed one-way ANOVA F-test to identify CpG sites with significant differences between survival groups
5. Applied FDR correction (Benjamini-Hochberg) to control false discovery rate
6. Selected only CpG sites that are significant after FDR correction

**Result:** From the original dataset, we selected CpG sites that show significant methylation differences between short and long survival patients.

---

### Next Steps: Associate each CpG to genes

In [22]:
cols_to_use = ['IlmnID', 'CHR', 'MAPINFO', 'UCSC_RefGene_Name', 'UCSC_RefGene_Group']
manifest = pd.read_csv('illumina-450k/manifest.csv', skiprows=7, usecols=cols_to_use)

manifest['Gene_symbol'] = manifest['UCSC_RefGene_Name'].str.split(';').str[0]

/var/folders/pl/097pyytj0n59vsv43sybymdw0000gn/T/ipykernel_5947/2191622491.py:2: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  manifest = pd.read_csv('illumina-450k/manifest.csv', skiprows=7, usecols=cols_to_use)


In [23]:
cpg_gene_mapping = manifest[['IlmnID', 'Gene_symbol', 'UCSC_RefGene_Group']].copy()
cpg_gene_mapping = cpg_gene_mapping.rename(columns={'IlmnID': 'CpG_ID'})

#Get the CpG IDs from your significant methylation data
significant_cpgs_list = dna_meth_significant.index.tolist()

In [24]:
cpg_annotations = cpg_gene_mapping[cpg_gene_mapping['CpG_ID'].isin(significant_cpgs_list)]
print(f"Total significant CpGs: {len(significant_cpgs_list)}")
print(f"CpGs with gene annotations: {len(cpg_annotations)}")
print(f"CpGs without annotations: {len(significant_cpgs_list) - len(cpg_annotations)}")

Total significant CpGs: 450
CpGs with gene annotations: 450
CpGs without annotations: 0


In [25]:
# Add gene information to your methylation data
dna_meth_with_genes = dna_meth_significant.merge(
    cpg_annotations[['CpG_ID', 'Gene_symbol']], 
    left_index=True, 
    right_on='CpG_ID'
)
# Set gene as index and drop CpG_ID column
dna_meth_with_genes = dna_meth_with_genes.set_index('Gene_symbol')
dna_meth_with_genes = dna_meth_with_genes.drop('CpG_ID', axis=1)

# Aggregate by gene (multiple CpGs per gene)
# Option 1: Mean (most common)
gene_methylation = dna_meth_with_genes.groupby('Gene_symbol').mean()

# Option 2: Median (more robust to outliers)
# gene_methylation = dna_meth_with_genes.groupby('Gene_symbol').median()

# Option 3: Max (some studies use this)
# gene_methylation = dna_meth_with_genes.groupby('Gene_symbol').max()

print(f"Original CpG sites: {dna_meth_significant.shape[0]}")
print(f"Unique genes: {gene_methylation.shape[0]}")
print(f"Gene methylation matrix shape: {gene_methylation.shape}")

Original CpG sites: 450
Unique genes: 293
Gene methylation matrix shape: (293, 785)


In [27]:
gene_methylation.T.head()

Gene_symbol,A2BP1,AACS,AARS,ABCA4,ABR,ACACA,ACLY,AEBP1,AFAP1L2,AGAP3,...,YAP1,ZC3H3,ZDHHC14,ZFAND3,ZFYVE19,ZNF710,ZNF783,ZNF792,ZNF804B,ZNF853
TCGA.3C.AAAU,0.163843,0.910137,0.927156,0.603047,0.533902,0.861460,0.938414,0.448345,0.798194,0.517299,...,0.700315,0.871959,0.120988,0.729374,0.876842,0.710936,0.331492,0.928117,0.064146,0.014104
TCGA.3C.AALI,0.155245,0.937484,0.859260,0.619868,0.698486,0.799271,0.864873,0.859636,0.743694,0.942081,...,0.465044,0.853709,0.195329,0.826238,0.855940,0.840341,0.618055,0.869470,0.090405,0.021827
TCGA.3C.AALJ,0.294967,0.358267,0.929430,0.698303,0.494446,0.780893,0.868957,0.854750,0.766747,0.478896,...,0.687297,0.829021,0.303417,0.571329,0.861390,0.783119,0.657276,0.889973,0.055197,0.020598
TCGA.3C.AALK,0.327437,0.893969,0.955224,0.703745,0.530417,0.798864,0.819986,0.848856,0.719758,0.792528,...,0.466580,0.880413,0.254600,0.724747,0.885453,0.732971,0.851391,0.858059,0.130095,0.017318
TCGA.4H.AAAK,0.111816,0.902272,0.926369,0.629701,0.516109,0.771839,0.816029,0.747271,0.673311,0.824992,...,0.533697,0.900456,0.485431,0.696999,0.890823,0.785495,0.728261,0.870601,0.043939,0.015583
